In [2]:
import math
import torch
from torch import nn
import torch.nn.functional as F

In [3]:
class LoRALinear(nn.Module):
    '''
    传入的参数包括输入的形状:(in_feature, out_feature)
    merge:表示最后合并时是否要将预训练的权重W加上去
    rank:表示最后A,B两个矩阵的秩是多少
    lora_alpha:控制放缩程度
    dropout:dropout的概率
    '''
    def __init__(self, in_features, out_features, merge, rank, lora_alpha, dropout):
        super(LoRALinear,self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.merge = merge
        self.rank = rank
        self.lora_alpha = lora_alpha
        self.dropout = dropout
        # 构建线性映射层
        self.linear = nn.Linear(in_features, out_features)
        if rank > 0:
            '''
            初始化矩阵A,B:
            B初始化为0矩阵，形状为out_feature * rank
            A使用高斯分布随机生成，之后再用海明初始化,形状为rank * in_feature
            '''
            # 这里要注意输出的形状是(out_features, in_features)
            self.lora_b = nn.Parameter(torch.zeros(out_features,rank))
            self.lora_a = nn.Parameter(torch.zeros(rank,in_features))
            
            # 放缩系数
            self.scale = lora_alpha / rank
            
            # 注意由于原始权重在微调过程中不需要发生变化，因此这里的梯度更新需要关闭
            self.linear.weight.requires_grad = False
            
        if dropout > 0:
            self.dropout = nn.Dropout(self.dropout)
        else:
            self.dropout = nn.Identity()
            
        # 对矩阵A使用海明初始化
        nn.init.kaiming_uniform_(self.lora_a, a = math.sqrt(5))
         # 对矩阵B进行初始化
        nn.init.zeros_(self.lora_b)
        
    def forward(self,x):
            if self.rank > 0 and self.merge:
                # 是否要使用原来的权重矩阵进行合并
                output = F.linear(x, self.linear.weight + self.lora_b @ self.lora_a * self.scale,self.linear.bias)
                output = self.dropout(output)
                return output
                # 不使用则直接返回
            else:
                return self.dropout(self.linear(x))
            

In [7]:
# 测试LoRA
batch_size = 32
rank = 8
seq_len = 128
in_feature = 768
out_feature = 512
lora_alpha = 16
dropout = 0.1

x = torch.rand(batch_size, seq_len, in_feature)

lora_layer1 = LoRALinear(in_features=in_feature, 
                        out_features=out_feature,
                        merge=False, 
                        rank=rank, 
                        lora_alpha=lora_alpha, 
                        dropout=0.1
                       )

# 计算,不进行权重合并
output1 = lora_layer1(x)
print(f"未合并下的形状:{output.shape}")

lora_layer2 = LoRALinear(in_features=in_feature, 
                        out_features=out_feature,
                        merge=True, 
                        rank=rank, 
                        lora_alpha=lora_alpha, 
                        dropout=0.1
                       )
output2 = lora_layer2(x)
print(f"合并下的形状:{output.shape}")

未合并下的形状:torch.Size([32, 128, 512])
合并下的形状:torch.Size([32, 128, 512])


In [14]:
print(output1)

tensor([[[-0.0000,  0.5222,  0.2378,  ..., -0.1057,  0.4039,  0.3277],
         [-0.4574,  0.5742,  0.1039,  ..., -0.2413,  0.0603,  0.3314],
         [ 0.0000,  0.1254,  0.1971,  ...,  0.1665,  0.6625,  0.0000],
         ...,
         [ 0.0316,  0.5578,  0.2248,  ...,  0.3078,  0.1096,  0.4226],
         [ 0.0448,  0.2869,  0.0925,  ...,  0.3370,  0.5535,  0.2748],
         [-0.0035,  0.2413,  0.2842,  ...,  0.0000,  0.4668,  0.3422]],

        [[-0.2785,  0.5076,  0.1401,  ..., -0.1365,  0.3457,  0.5676],
         [-0.1493,  0.5515,  0.0691,  ..., -0.0620,  0.6957,  0.3836],
         [ 0.0473,  0.0916,  0.5987,  ...,  0.2253,  0.3939,  0.4331],
         ...,
         [-0.0265,  0.0000,  0.0000,  ...,  0.2615,  0.5336,  0.1296],
         [-0.2061,  0.5291,  0.4049,  ...,  0.0541,  0.2517,  0.4297],
         [ 0.0538,  0.1212,  0.1681,  ...,  0.4278,  0.4160,  0.3909]],

        [[ 0.0205,  0.6388, -0.0435,  ..., -0.0279,  0.7641,  0.4107],
         [ 0.1680,  0.8384,  0.2121,  ..., -0

In [9]:
print(output2)

tensor([[[-0.2270, -0.4955, -0.1184,  ...,  0.5987,  0.3735, -0.6927],
         [-0.3092, -0.3320, -0.0000,  ...,  0.5109,  0.4194, -0.5182],
         [-0.4995, -0.3206, -0.2073,  ...,  0.2197,  0.0816, -0.5132],
         ...,
         [ 0.0167, -0.1736, -0.1482,  ...,  0.3943,  0.2983, -0.2241],
         [-0.4321, -0.2896, -0.2481,  ...,  0.4684,  0.4278, -0.6082],
         [-0.5415, -0.0000, -0.1736,  ...,  0.4328,  0.1340, -0.5286]],

        [[-0.3015, -0.3545, -0.0819,  ...,  0.5815, -0.0077, -0.7171],
         [-0.2553, -0.2416, -0.0000,  ...,  0.7672,  0.3426,  0.0168],
         [-0.2759, -0.0000, -0.2443,  ...,  0.3566,  0.3434, -0.3174],
         ...,
         [-0.2934, -0.1521, -0.1002,  ...,  0.9431,  0.1394, -0.0000],
         [-0.3958, -0.2055, -0.0000,  ...,  0.8282,  0.4075, -0.8268],
         [-0.2766, -0.0000, -0.0873,  ...,  0.5127,  0.4409, -0.3704]],

        [[-0.4124, -0.0206, -0.0737,  ...,  0.6060,  0.5139, -0.3447],
         [-0.3598, -0.1998, -0.0681,  ...,  0

In [13]:
# 这个就是ΔW * x
print(output2-output1)

tensor([[[-0.2270, -1.0177, -0.3563,  ...,  0.7045, -0.0304, -1.0204],
         [ 0.1482, -0.9062, -0.1039,  ...,  0.7521,  0.3592, -0.8497],
         [-0.4995, -0.4460, -0.4044,  ...,  0.0531, -0.5809, -0.5132],
         ...,
         [-0.0149, -0.7314, -0.3730,  ...,  0.0866,  0.1887, -0.6467],
         [-0.4769, -0.5765, -0.3406,  ...,  0.1314, -0.1258, -0.8830],
         [-0.5381, -0.2413, -0.4578,  ...,  0.4328, -0.3328, -0.8708]],

        [[-0.0230, -0.8622, -0.2220,  ...,  0.7180, -0.3534, -1.2848],
         [-0.1060, -0.7931, -0.0691,  ...,  0.8293, -0.3530, -0.3668],
         [-0.3233, -0.0916, -0.8430,  ...,  0.1313, -0.0505, -0.7505],
         ...,
         [-0.2669, -0.1521, -0.1002,  ...,  0.6817, -0.3942, -0.1296],
         [-0.1897, -0.7346, -0.4049,  ...,  0.7741,  0.1557, -1.2565],
         [-0.3303, -0.1212, -0.2554,  ...,  0.0849,  0.0248, -0.7613]],

        [[-0.4329, -0.6595, -0.0302,  ...,  0.6339, -0.2501, -0.7553],
         [-0.5278, -1.0382, -0.2802,  ...,  0